# Relaxationszeiten aus exponentiellen Verlaeufen

Dieses Notebook bestimmt Zeitkonstanten aus den zeitlichen Verlaeufen von

- der direkt bestimmten Stufenhoehe `delta epsilon'` (`Eps_real_Step_Height`),
- der aus dem Fit bestimmten dielektrischen Staerke `delta epsilon` (`de`),
- der Peakposition `omega_p`.

Gefittet wird pro Material, Temperatur, Modus und Observable ein exponentieller Verlauf

`y(t) = y_inf + A * exp(-t / tau)`

wobei `tau` die gesuchte Relaxationszeit ist.

Fuer `delta epsilon'` aus der Stufenhoehe wird je nach Modus geteilt: Bei Absorption laeuft der erste Fit vom automatisch erkannten Knick bis zum Minimum und der zweite vom Minimum bis zum Ende. Bei Desorption wird `delta epsilon'` aus der Stufenhoehe fuer 50, 60 und 70 C vom Vor-Null-Bereich bis in den Bereich nach dem Maximum mit einer Summe aus drei exponentiellen Saettigungstermen gefittet. Fuer `delta epsilon` aus dem Fit wird normalerweise ein Fit ueber den ausgewaehlten Zeitbereich verwendet; fuer 70 C Abs wird `delta epsilon` aus dem Fit bei 1200 s in zwei Fits geteilt, fuer 50 C Des wird nur bis 3000 s gefittet, fuer 60 C Des bei 900 s geteilt, fuer 70 C Des bei 350 s geteilt. Die Fit-Parameter `delta epsilon` (`de`) werden nur fuer 50, 60 und 70 C ausgewertet. Fuer `delta epsilon'` aus der Stufenhoehe bleiben die Absorptionsmessungen bei 80 und 90 C zusaetzlich enthalten; bei 80 C endet der zweite Absorptions-Fit am anschliessenden Maximum. `omega_p` wird fuer die Absorptionsmessungen bei 50, 60 und 70 C gefittet und im Arrhenius-Plot als eigene Serie gezeigt.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter

plt.style.use("default")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "code" else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
PLOT_DIR = RESULTS_DIR / "relaxation_time_plots"

PROJECT_ROOT, RESULTS_DIR

## Einstellungen

`FIT_AFTER_SWITCH_ONLY = True` verwendet nur Messpunkte ab dem Schaltpunkt (`Time_Relative_s >= 0`). Wenn du den gesamten Verlauf fitten moechtest, setze den Wert auf `False`.

`FIT_TEMPERATURES_C = [50, 60, 70]` beschraenkt die normalen Relaxations-Fits auf 50, 60 und 70 C. `EXTRA_STEP_HEIGHT_ABS_TEMPERATURES_C = [80, 90]` nimmt 80 und 90 C nur fuer `delta epsilon'` aus der Stufenhoehe im Abs-Modus wieder dazu.

`FIT_OBSERVABLES` legt fest, welche Observables gefittet werden. `omega_p` wird nur fuer Absorption bei 50, 60 und 70 C verwendet.

`CORRECT_STEP_HEIGHT_ABS_JUMPS = True` korrigiert die angegebenen Spruenge in `delta epsilon'` fuer Absorption nur innerhalb des Notebooks. Die CSV-Rohdaten werden nicht veraendert.

In [ ]:
USE_PILOT_FITS = False
FIT_AFTER_SWITCH_ONLY = True
MIN_TIME_S = 0.0
MIN_POINTS_PER_FIT = 6
FIT_TEMPERATURES_C = [50, 60, 70]
EXTRA_STEP_HEIGHT_ABS_TEMPERATURES_C = [80, 90]
FIT_OBSERVABLES = ["delta_eps_real_step_height", "delta_eps_fit_de", "omega_p"]
CORRECT_STEP_HEIGHT_ABS_JUMPS = True
STEP_HEIGHT_ABS_JUMP_CORRECTIONS = [
    {"material": "PEI5mgmL", "temperature_c": 70.0, "start_s": 3310.0, "end_s": 3390.0, "fit_window_s": 300.0},
    {"material": "PEI5mgmL", "temperature_c": 80.0, "start_s": 4650.0, "end_s": 4730.0, "fit_window_s": 300.0},
    {"material": "PEI5mgmL", "temperature_c": 90.0, "start_s": 4250.0, "end_s": 4330.0, "fit_window_s": 300.0},
]

# Rohdaten nicht global einschraenken; die Fit-Filter trennen de und Stepheight.
MATERIALS = None
TEMPERATURES_C = None
MODES = None

FIT_FILE_PATTERN = "fit_parameters_pilot_*.csv" if USE_PILOT_FITS else "fit_parameters_*.csv ohne pilot"
FIT_FILE_PATTERN

In [ ]:
def parse_temperature_c(value):
    match = re.search(r"(\d+(?:\.\d+)?)", str(value))
    return float(match.group(1)) if match else np.nan


def parse_fit_filename(path):
    match = re.match(
        r"fit_parameters(?:_pilot)?_(?P<material>.+?)_(?P<temperature>\d+)C_(?P<mode>Abs|Des)\.csv$",
        path.name,
    )
    if not match:
        raise ValueError(f"Dateiname passt nicht zum erwarteten Muster: {path.name}")
    return {
        "Material": match.group("material"),
        "Temperature_C": float(match.group("temperature")),
        "Mode": match.group("mode"),
    }


def load_direct_step_heights():
    all_series = RESULTS_DIR / "step_height_eps_real_all_series.csv"
    if all_series.exists():
        df = pd.read_csv(all_series)
    else:
        files = sorted(RESULTS_DIR.glob("step_height_eps_real_*C_*.csv"))
        df = pd.concat((pd.read_csv(path) for path in files), ignore_index=True)

    df = df.copy()
    df["Temperature_C"] = df["Temperature"].map(parse_temperature_c)
    return df


def load_fit_parameters():
    if USE_PILOT_FITS:
        files = sorted(RESULTS_DIR.glob("fit_parameters_pilot_*.csv"))
    else:
        files = sorted(
            path for path in RESULTS_DIR.glob("fit_parameters_*.csv")
            if not path.name.startswith("fit_parameters_pilot_")
        )
    if not files:
        raise FileNotFoundError(f"Keine Fitparameter-Dateien gefunden: {FIT_FILE_PATTERN}")

    frames = []
    for path in files:
        df = pd.read_csv(path)
        for key, value in parse_fit_filename(path).items():
            df[key] = value
        df["Source_File"] = path.name
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


direct_df = load_direct_step_heights()
fit_df = load_fit_parameters()

direct_df.head(), fit_df.head()

In [ ]:
def apply_selection(df):
    selected = df.copy()
    if MATERIALS is not None:
        selected = selected[selected["Material"].isin(MATERIALS)]
    if TEMPERATURES_C is not None:
        selected = selected[selected["Temperature_C"].isin(TEMPERATURES_C)]
    if MODES is not None:
        selected = selected[selected["Mode"].isin(MODES)]
    return selected


def to_observable_frame(df, value_column, observable, source):
    required = ["Material", "Temperature_C", "Mode", "Time_Relative_s", value_column]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise KeyError(f"Fehlende Spalten fuer {observable}: {missing}")

    out = df[required].copy()
    out = out.rename(columns={value_column: "value"})
    out["observable"] = observable
    out["source"] = source
    return out


direct_selected = apply_selection(direct_df)
fit_selected = apply_selection(fit_df)

if "Accepted" in fit_selected.columns:
    fit_selected = fit_selected[fit_selected["Accepted"].fillna(False).astype(bool)]
if "Fit_Success" in fit_selected.columns:
    fit_selected = fit_selected[fit_selected["Fit_Success"].fillna(False).astype(bool)]

observables = [
    to_observable_frame(direct_selected, "Eps_real_Step_Height", "delta_eps_real_step_height", "step_height"),
    to_observable_frame(fit_selected, "de", "delta_eps_fit_de", "fit"),
    to_observable_frame(fit_selected, "omega_p", "omega_p", "fit"),
]

data_long = pd.concat(observables, ignore_index=True)
data_long = data_long.replace([np.inf, -np.inf], np.nan).dropna(subset=["Time_Relative_s", "value"])


def correct_step_height_abs_jumps(df, corrections):
    corrected = df.copy()
    info_rows = []

    for correction in corrections:
        material = correction["material"]
        temperature_c = float(correction["temperature_c"])
        start = float(correction["start_s"])
        end = float(correction["end_s"])
        window = float(correction.get("fit_window_s", 300.0))

        mask = (
            (corrected["Material"] == material)
            & (corrected["Temperature_C"] == temperature_c)
            & (corrected["Mode"] == "Abs")
            & (corrected["observable"] == "delta_eps_real_step_height")
        )
        target = corrected.loc[mask].sort_values("Time_Relative_s")
        if target.empty:
            continue

        pre = target[(target["Time_Relative_s"] >= start - window) & (target["Time_Relative_s"] < start)]
        post = target[(target["Time_Relative_s"] > end) & (target["Time_Relative_s"] <= end + window)]
        if len(pre) < 2 or len(post) < 2:
            info_rows.append({
                "Material": material,
                "Temperature_C": temperature_c,
                "Mode": "Abs",
                "observable": "delta_eps_real_step_height",
                "jump_start_s": start,
                "jump_end_s": end,
                "fit_window_s": window,
                "correction_status": "too_few_points",
            })
            continue

        pre_fit = np.polyfit(pre["Time_Relative_s"], pre["value"], 1)
        post_fit = np.polyfit(post["Time_Relative_s"], post["value"], 1)
        offset = float(np.polyval(post_fit, end) - np.polyval(pre_fit, end))

        ramp_mask = mask & (corrected["Time_Relative_s"] >= start) & (corrected["Time_Relative_s"] <= end)
        after_mask = mask & (corrected["Time_Relative_s"] > end)
        ramp_times = corrected.loc[ramp_mask, "Time_Relative_s"]
        ramp_start_value = float(np.polyval(pre_fit, start))
        ramp_end_value = float(np.polyval(post_fit, end) - offset)
        corrected.loc[ramp_mask, "value"] = np.interp(ramp_times, [start, end], [ramp_start_value, ramp_end_value])
        corrected.loc[after_mask, "value"] = corrected.loc[after_mask, "value"] - offset

        info_rows.append({
            "Material": material,
            "Temperature_C": temperature_c,
            "Mode": "Abs",
            "observable": "delta_eps_real_step_height",
            "jump_start_s": start,
            "jump_end_s": end,
            "fit_window_s": window,
            "subtracted_offset": offset,
            "pre_slope": float(pre_fit[0]),
            "post_slope": float(post_fit[0]),
            "correction_status": "ok",
        })

    return corrected, pd.DataFrame(info_rows)


jump_correction_info = pd.DataFrame()
if CORRECT_STEP_HEIGHT_ABS_JUMPS:
    data_long, jump_correction_info = correct_step_height_abs_jumps(data_long, STEP_HEIGHT_ABS_JUMP_CORRECTIONS)

data_long = data_long.sort_values(["Material", "Temperature_C", "Mode", "observable", "Time_Relative_s"])

if not jump_correction_info.empty:
    display(jump_correction_info)

data_long.groupby(["Material", "Temperature_C", "Mode", "observable"]).size().rename("n_points").reset_index()

## Delta epsilon darstellen

In diesem Block koennen `delta epsilon'` aus der Stufenhoehe und `delta epsilon` aus dem Fit dargestellt werden. Von den drei Parametern `Variable`, `Modus` und `Temperatur` muessen immer genau zwei festgehalten werden. Der dritte Parameter wird dann im Plot variiert.

In [ ]:
DELTA_OBSERVABLES = {
    "delta_eps_real_step_height": "Delta epsilon' aus Stufenhoehe",
    "delta_eps_fit_de": "Delta epsilon aus Fit (de)",
}


def is_multi_selection(value):
    return isinstance(value, (list, tuple, set, np.ndarray, pd.Series)) and not isinstance(value, str)


def selection_values(value, *, as_float=False):
    if value is None:
        return None
    values = list(value) if is_multi_selection(value) else [value]
    if as_float:
        return [float(item) for item in values]
    return values


def selection_is_varied(value):
    return value is None or (is_multi_selection(value) and len(value) != 1)


def format_selection(value, unit=""):
    if value is None:
        return "alle"
    values = list(value) if is_multi_selection(value) else [value]
    formatted = []
    for item in values:
        if isinstance(item, (int, float, np.integer, np.floating)):
            formatted.append(f"{float(item):g}{unit}")
        else:
            formatted.append(f"{item}{unit}")
    return ", ".join(formatted)


def normalize_axis_scale(scale, *, log_flag=False):
    if log_flag:
        return "log"
    if scale is None:
        return "linear"
    scale = str(scale).lower()
    if scale not in {"linear", "log", "symlog"}:
        raise ValueError("Achsen-Skala muss 'linear', 'log' oder 'symlog' sein.")
    return scale


def plot_delta_epsilon(
    fixed_variable=None,
    fixed_mode=None,
    fixed_temperature_c=None,
    x_scale="linear",
    y_scale="linear",
    log_x=False,
    log_y=False,
    symlog_linthresh_x=1.0,
    symlog_linthresh_y=1e-3,
    xlim=None,
    ylim=None,
    figsize=(7.2, 4.4),
):
    x_scale = normalize_axis_scale(x_scale, log_flag=log_x)
    y_scale = normalize_axis_scale(y_scale, log_flag=log_y)
    fixed = {
        "Variable": fixed_variable,
        "Modus": fixed_mode,
        "Temperatur": fixed_temperature_c,
    }
    varied_parameters = [name for name, value in fixed.items() if selection_is_varied(value)]
    if len(varied_parameters) > 1:
        raise ValueError("Bitte hoechstens einen Parameter variieren: None fuer alle Werte oder Liste/Tupel fuer eine Auswahl.")

    plot_df = data_long[data_long["observable"].isin(DELTA_OBSERVABLES)].copy()
    variable_values = selection_values(fixed_variable)
    mode_values = selection_values(fixed_mode)
    temperature_values = selection_values(fixed_temperature_c, as_float=True)

    if variable_values is not None:
        plot_df = plot_df[plot_df["observable"].isin(variable_values)]
    if mode_values is not None:
        plot_df = plot_df[plot_df["Mode"].isin(mode_values)]
    if temperature_values is not None:
        plot_df = plot_df[plot_df["Temperature_C"].isin(temperature_values)]

    if plot_df.empty:
        raise ValueError("Keine Daten fuer diese Auswahl gefunden.")
    if x_scale == "log":
        plot_df = plot_df[plot_df["Time_Relative_s"] > 0]
    if y_scale == "log":
        plot_df = plot_df[plot_df["value"] > 0]
    if plot_df.empty:
        raise ValueError("Keine positiven Datenpunkte fuer die gewaehlte logarithmische Darstellung gefunden.")

    if len(varied_parameters) == 0:
        vary_column = None
        vary_label = "keine Variation"
    elif selection_is_varied(fixed_variable):
        vary_column = "observable"
        vary_label = "Variable"
    elif selection_is_varied(fixed_mode):
        vary_column = "Mode"
        vary_label = "Modus"
    else:
        vary_column = "Temperature_C"
        vary_label = "Temperatur"

    fig, ax = plt.subplots(figsize=figsize)
    if vary_column is None:
        group_columns = ["Material"] if "Material" in plot_df.columns and plot_df["Material"].nunique() > 1 else []
    else:
        group_columns = [vary_column]
        if "Material" in plot_df.columns and plot_df["Material"].nunique() > 1:
            group_columns = ["Material", vary_column]

    grouped = [("Auswahl", plot_df)] if not group_columns else plot_df.groupby(group_columns)
    for key, group in grouped:
        group = group.sort_values("Time_Relative_s")
        key_parts = key if isinstance(key, tuple) else (key,)
        vary_value = key_parts[-1]
        if vary_column is None:
            label = "Auswahl"
        elif vary_column == "observable":
            label = DELTA_OBSERVABLES.get(vary_value, vary_value)
        elif vary_column == "Temperature_C":
            label = f"{float(vary_value):g} C"
        else:
            label = str(vary_value)

        if len(key_parts) > 1:
            label = " | ".join(str(part) for part in key_parts[:-1] + (label,))

        ax.plot(group["Time_Relative_s"], group["value"], marker=".", linewidth=1.2, markersize=2.5, label=label)

    title_parts = []
    for name, value in fixed.items():
        if name == "Temperatur":
            title_parts.append(f"{name}: {format_selection(value, ' C')}")
        else:
            title_parts.append(f"{name}: {format_selection(value)}")
    if vary_column is None:
        ax.set_title("Delta epsilon | " + ", ".join(title_parts))
    else:
        ax.set_title("Delta epsilon, variiert: " + vary_label + " | " + ", ".join(title_parts))
    ax.set_xlabel("relative Zeit / s")
    ax.set_ylabel("Delta epsilon")
    if x_scale == "log":
        ax.set_xscale("log")
    elif x_scale == "symlog":
        ax.set_xscale("symlog", linthresh=symlog_linthresh_x)
    if y_scale == "log":
        ax.set_yscale("log")
    elif y_scale == "symlog":
        ax.set_yscale("symlog", linthresh=symlog_linthresh_y)
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.legend(title=None if vary_column is None else vary_label)
    fig.tight_layout()
    return fig, ax


# Beispiele: zwei Parameter skalar festhalten, der dritte wird variiert.
# plot_delta_epsilon(fixed_variable="delta_eps_fit_de", fixed_mode="Abs")
# plot_delta_epsilon(fixed_variable="delta_eps_real_step_height", fixed_temperature_c=50)
# plot_delta_epsilon(fixed_mode="Abs", fixed_temperature_c=50)
# plot_delta_epsilon(fixed_variable="delta_eps_real_step_height", fixed_mode="Des", fixed_temperature_c=(50, 60, 70), x_scale="log")
# plot_delta_epsilon(fixed_variable="delta_eps_fit_de", fixed_mode="Abs", fixed_temperature_c=None, x_scale="log", y_scale="log")
# plot_delta_epsilon(fixed_variable="delta_eps_real_step_height", fixed_mode="Abs", fixed_temperature_c=None, x_scale="symlog")

In [ ]:
VARIABLE = "delta_eps_real_step_height"  # "delta_eps_real_step_height" oder "delta_eps_fit_de"
MODUS = "Abs"  # "Abs", "Des" oder None
TEMPERATUR_C = None  # z.B. 50, (50, 60, 70) oder None

X_SCALE = "linear"  # "linear", "log" oder "symlog"
Y_SCALE = "linear"  # "linear", "log" oder "symlog"
X_LIM = None  # z.B. (-100, 5000) oder None fuer automatisch
Y_LIM = None  # z.B. (2.0, 4.5) oder None fuer automatisch
FIGSIZE = (12, 8)  # Breite, Hoehe in inch

plot_delta_epsilon(
    fixed_variable=VARIABLE,
    fixed_mode=MODUS,
    fixed_temperature_c=TEMPERATUR_C,
    x_scale=X_SCALE,
    y_scale=Y_SCALE,
    xlim=X_LIM,
    ylim=Y_LIM,
    figsize=FIGSIZE,
);

In [ ]:
def exp_model(t, y_inf, amplitude, tau):
    return y_inf + amplitude * np.exp(-t / tau)


def triple_exp_saturation_model(t, y0, amplitude1, tau1, amplitude2, tau2, amplitude3, tau3):
    return (
        y0
        + amplitude1 * (1 - np.exp(-t / tau1))
        + amplitude2 * (1 - np.exp(-t / tau2))
        + amplitude3 * (1 - np.exp(-t / tau3))
    )


STEP_HEIGHT_OBSERVABLE = "delta_eps_real_step_height"
FIT_DE_OBSERVABLE = "delta_eps_fit_de"
OMEGA_P_OBSERVABLE = "omega_p"
TRIPLE_EXP_DES_STEP_TEMPERATURES_C = [50, 60, 70]
DE_70_ABS_SPLIT_TIME_S = 1200.0
DE_70_DES_SPLIT_TIME_S = 350.0
DE_50_DES_END_TIME_S = 3000.0
DE_60_DES_SPLIT_TIME_S = 900.0
STEP_HEIGHT_50_DES_RANGES_S = [(0.0, 75.0), (75.0, 425.0)]
STEP_HEIGHT_60_DES_RANGES_S = [(0.0, 200.0), (200.0, 325.0)]


def prepare_fit_group(group, after_switch_only=None):
    group = group.sort_values("Time_Relative_s").copy()
    if after_switch_only is None:
        after_switch_only = FIT_AFTER_SWITCH_ONLY
    if after_switch_only:
        group = group[group["Time_Relative_s"] >= MIN_TIME_S]
    return group.dropna(subset=["Time_Relative_s", "value"])


def find_kink_before_minimum(group, min_time):
    before_minimum = group[group["Time_Relative_s"] <= min_time].copy()
    if len(before_minimum) < max(MIN_POINTS_PER_FIT, 7):
        return float(before_minimum["Time_Relative_s"].min()), float(before_minimum["value"].iloc[0])

    t = before_minimum["Time_Relative_s"].to_numpy(dtype=float)
    y = before_minimum["value"].to_numpy(dtype=float)
    y_smooth = smooth_for_inflection(y)
    dt = t[-1] - t[0]
    dy = y_smooth[-1] - y_smooth[0]
    norm = np.hypot(dt, dy)
    if norm == 0:
        return float(t[0]), float(y[0])

    distances = np.abs(dt * (y_smooth[0] - y_smooth) - (t[0] - t) * dy) / norm
    margin = max(2, int(0.05 * len(before_minimum)))
    if len(distances) > 2 * margin:
        search_distances = distances[margin:-margin]
        kink_idx = margin + int(np.nanargmax(search_distances))
    else:
        kink_idx = int(np.nanargmax(distances))
    return float(t[kink_idx]), float(y[kink_idx])


def split_at_minimum(group):
    group = prepare_fit_group(group)
    if group.empty:
        return [("to_minimum", group, {}), ("from_minimum", group, {})]

    min_idx = group["value"].idxmin()
    min_time = float(group.loc[min_idx, "Time_Relative_s"])
    min_value = float(group.loc[min_idx, "value"])
    kink_time, kink_value = find_kink_before_minimum(group, min_time)
    metadata = {
        "minimum_time_s": min_time,
        "minimum_value": min_value,
        "kink_time_s": kink_time,
        "kink_value": kink_value,
        "split_point_1_s": min_time,
        "split_point_1_value": min_value,
        "split_point_1_type": "minimum",
        "split_point_2_s": kink_time,
        "split_point_2_value": kink_value,
        "split_point_2_type": "kink",
    }

    return [
        ("to_minimum", group[(group["Time_Relative_s"] >= kink_time) & (group["Time_Relative_s"] <= min_time)], metadata),
        ("from_minimum", group[group["Time_Relative_s"] >= min_time], metadata),
    ]


def smooth_for_inflection(y):
    n = len(y)
    if n < 7:
        return y
    window = min(301, max(7, int(n * 0.05)))
    if window % 2 == 0:
        window += 1
    if window >= n:
        window = n - 1 if n % 2 == 0 else n
    if window < 7:
        return y
    return savgol_filter(y, window_length=window, polyorder=3)


def find_inflection_times(group):
    group = prepare_fit_group(group)
    if len(group) < MIN_POINTS_PER_FIT:
        return [], group

    t = group["Time_Relative_s"].to_numpy(dtype=float)
    y = group["value"].to_numpy(dtype=float)
    y_smooth = smooth_for_inflection(y)
    first_derivative = np.gradient(y_smooth, t)
    second_derivative = np.gradient(first_derivative, t)

    signs = np.sign(second_derivative)
    nonzero = signs != 0
    if nonzero.any():
        signs = pd.Series(signs).replace(0, np.nan).ffill().bfill().to_numpy()

    change_indices = np.where(np.diff(signs) != 0)[0] + 1
    margin = max(2, int(0.03 * len(group)))
    change_indices = [idx for idx in change_indices if margin <= idx < len(group) - margin]

    selected = []
    min_distance = max(2, int(0.08 * len(group)))
    for idx in change_indices:
        if not selected or idx - selected[-1] >= min_distance:
            selected.append(idx)
        if len(selected) == 2:
            break

    return [(float(t[idx]), float(y[idx])) for idx in selected], group


def split_at_des_inflections(group):
    inflections, group = find_inflection_times(group)
    if len(inflections) < 2:
        metadata = {
            "minimum_time_s": np.nan,
            "minimum_value": np.nan,
            "split_point_1_s": np.nan,
            "split_point_1_value": np.nan,
            "split_point_1_type": "inflection",
            "split_point_2_s": np.nan,
            "split_point_2_value": np.nan,
            "split_point_2_type": "inflection",
        }
        return [("to_first_inflection", group.iloc[0:0], metadata), ("first_to_second_inflection", group.iloc[0:0], metadata)]

    (first_time, first_value), (second_time, second_value) = inflections[:2]
    metadata = {
        "minimum_time_s": np.nan,
        "minimum_value": np.nan,
        "split_point_1_s": first_time,
        "split_point_1_value": first_value,
        "split_point_1_type": "inflection",
        "split_point_2_s": second_time,
        "split_point_2_value": second_value,
        "split_point_2_type": "inflection",
    }
    return [
        ("to_first_inflection", group[group["Time_Relative_s"] <= first_time], metadata),
        ("first_to_second_inflection", group[(group["Time_Relative_s"] >= first_time) & (group["Time_Relative_s"] <= second_time)], metadata),
    ]


def split_step_height_group(group, mode):
    if mode == "Des":
        return split_at_des_inflections(group)
    return split_at_minimum(group)


def split_step_height_80_abs_to_maximum(group):
    group = prepare_fit_group(group)
    if group.empty:
        return [("to_minimum", group, {}), ("from_minimum", group, {})]

    min_idx = group["value"].idxmin()
    min_time = float(group.loc[min_idx, "Time_Relative_s"])
    min_value = float(group.loc[min_idx, "value"])
    kink_time, kink_value = find_kink_before_minimum(group, min_time)
    after_minimum = group[group["Time_Relative_s"] >= min_time]
    max_idx = after_minimum["value"].idxmax()
    max_time = float(group.loc[max_idx, "Time_Relative_s"])
    max_value = float(group.loc[max_idx, "value"])
    metadata = {
        "minimum_time_s": min_time,
        "minimum_value": min_value,
        "kink_time_s": kink_time,
        "kink_value": kink_value,
        "split_point_1_s": min_time,
        "split_point_1_value": min_value,
        "split_point_1_type": "minimum",
        "split_point_2_s": max_time,
        "split_point_2_value": max_value,
        "split_point_2_type": "maximum",
        "split_point_3_s": kink_time,
        "split_point_3_value": kink_value,
        "split_point_3_type": "kink",
    }
    return [
        ("to_minimum", group[(group["Time_Relative_s"] >= kink_time) & (group["Time_Relative_s"] <= min_time)], metadata),
        ("from_minimum", group[(group["Time_Relative_s"] >= min_time) & (group["Time_Relative_s"] <= max_time)], metadata),
    ]


def fit_step_height_des_post_maximum_window(group):
    group = prepare_fit_group(group, after_switch_only=False)
    if group.empty:
        return [("triple_exp_post_maximum", group, {})]

    max_idx = group["value"].idxmax()
    max_time = float(group.loc[max_idx, "Time_Relative_s"])
    max_value = float(group.loc[max_idx, "value"])
    last_time = float(group["Time_Relative_s"].max())
    end_time = min(last_time, max_time * 3) if max_time > 0 else last_time
    metadata = {
        "minimum_time_s": np.nan,
        "minimum_value": np.nan,
        "split_point_1_s": max_time,
        "split_point_1_value": max_value,
        "split_point_1_type": "maximum",
        "split_point_2_s": np.nan,
        "split_point_2_value": np.nan,
        "split_point_2_type": "",
    }
    return [("triple_exp_post_maximum", group[group["Time_Relative_s"] <= end_time], metadata)]


def split_at_time(group, split_time_s):
    group = prepare_fit_group(group)
    if group.empty:
        return [("to_1200s", group, {}), ("from_1200s", group, {})]

    nearest_idx = (group["Time_Relative_s"] - split_time_s).abs().idxmin()
    split_value = float(group.loc[nearest_idx, "value"])
    metadata = {
        "minimum_time_s": np.nan,
        "minimum_value": np.nan,
        "split_point_1_s": float(split_time_s),
        "split_point_1_value": split_value,
        "split_point_1_type": "manual_time",
        "split_point_2_s": np.nan,
        "split_point_2_value": np.nan,
        "split_point_2_type": "",
    }
    label_time = f"{split_time_s:g}s"
    return [
        (f"to_{label_time}", group[group["Time_Relative_s"] <= split_time_s], metadata),
        (f"from_{label_time}", group[group["Time_Relative_s"] >= split_time_s], metadata),
    ]


def fit_until_time(group, end_time_s):
    group = prepare_fit_group(group)
    if group.empty:
        return [(f"to_{end_time_s:g}s", group, {})]

    nearest_idx = (group["Time_Relative_s"] - end_time_s).abs().idxmin()
    end_value = float(group.loc[nearest_idx, "value"])
    metadata = {
        "minimum_time_s": np.nan,
        "minimum_value": np.nan,
        "split_point_1_s": float(end_time_s),
        "split_point_1_value": end_value,
        "split_point_1_type": "manual_time",
        "split_point_2_s": np.nan,
        "split_point_2_value": np.nan,
        "split_point_2_type": "",
    }
    return [(f"to_{end_time_s:g}s", group[group["Time_Relative_s"] <= end_time_s], metadata)]


def split_into_time_ranges(group, ranges_s):
    group = prepare_fit_group(group)
    if group.empty:
        return [(f"{start:g}s_to_{end:g}s", group, {}) for start, end in ranges_s]

    split_times = sorted({float(time) for time_range in ranges_s for time in time_range})
    metadata = {
        "minimum_time_s": np.nan,
        "minimum_value": np.nan,
        "split_point_1_s": split_times[1] if len(split_times) > 1 else np.nan,
        "split_point_1_value": np.nan,
        "split_point_1_type": "manual_time",
        "split_point_2_s": split_times[2] if len(split_times) > 2 else np.nan,
        "split_point_2_value": np.nan,
        "split_point_2_type": "manual_time" if len(split_times) > 2 else "",
    }
    for point_key, value_key in [("split_point_1_s", "split_point_1_value"), ("split_point_2_s", "split_point_2_value")]:
        split_time = metadata[point_key]
        if pd.notna(split_time):
            nearest_idx = (group["Time_Relative_s"] - split_time).abs().idxmin()
            metadata[value_key] = float(group.loc[nearest_idx, "value"])

    segments = []
    for start_time, end_time in ranges_s:
        segment = group[(group["Time_Relative_s"] >= start_time) & (group["Time_Relative_s"] <= end_time)]
        segments.append((f"{start_time:g}s_to_{end_time:g}s", segment, metadata))
    return segments


def should_split_de_70_abs(row):
    return (
        row["observable"] == FIT_DE_OBSERVABLE
        and row["Mode"] == "Abs"
        and float(row["Temperature_C"]) == 70.0
    )


def should_fit_de_50_des_to_3000(row):
    return (
        row["observable"] == FIT_DE_OBSERVABLE
        and row["Mode"] == "Des"
        and float(row["Temperature_C"]) == 50.0
    )


def should_split_de_60_des(row):
    return (
        row["observable"] == FIT_DE_OBSERVABLE
        and row["Mode"] == "Des"
        and float(row["Temperature_C"]) == 60.0
    )


def should_split_de_70_des(row):
    return (
        row["observable"] == FIT_DE_OBSERVABLE
        and row["Mode"] == "Des"
        and float(row["Temperature_C"]) == 70.0
    )


def should_fit_step_height_des_triple_exp(row):
    return (
        row["observable"] == STEP_HEIGHT_OBSERVABLE
        and row["Mode"] == "Des"
        and float(row["Temperature_C"]) in TRIPLE_EXP_DES_STEP_TEMPERATURES_C
    )


def should_split_step_height_80_abs_to_maximum(row):
    return (
        row["observable"] == STEP_HEIGHT_OBSERVABLE
        and row["Mode"] == "Abs"
        and float(row["Temperature_C"]) == 80.0
    )


def fit_exponential(group):
    group = prepare_fit_group(group)
    if len(group) < MIN_POINTS_PER_FIT:
        return None, group, "too_few_points"

    t_raw = group["Time_Relative_s"].to_numpy(dtype=float)
    y = group["value"].to_numpy(dtype=float)
    t = t_raw - np.nanmin(t_raw)

    if np.nanmax(t) <= 0 or np.nanstd(y) == 0:
        return None, group, "not_enough_variation"

    tail_n = max(3, len(y) // 5)
    y_inf0 = float(np.nanmedian(y[-tail_n:]))
    amplitude0 = float(y[0] - y_inf0)
    if amplitude0 == 0:
        amplitude0 = float(np.nanmax(y) - np.nanmin(y))
    tau0 = max(float(np.nanmax(t) / 3), 1e-9)

    y_span = max(float(np.nanmax(y) - np.nanmin(y)), 1e-12)
    lower = [float(np.nanmin(y) - 5 * y_span), -10 * y_span, 1e-9]
    upper = [float(np.nanmax(y) + 5 * y_span), 10 * y_span, max(float(np.nanmax(t) * 100), 1.0)]

    try:
        popt, pcov = curve_fit(
            exp_model,
            t,
            y,
            p0=[y_inf0, amplitude0, tau0],
            bounds=(lower, upper),
            maxfev=20000,
        )
    except Exception as exc:
        return None, group, f"fit_failed: {exc}"

    y_fit = exp_model(t, *popt)
    residuals = y - y_fit
    ss_res = float(np.sum(residuals**2))
    ss_tot = float(np.sum((y - np.mean(y))**2))
    r2 = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot
    rmse = float(np.sqrt(np.mean(residuals**2)))

    perr = np.full(3, np.nan)
    if pcov is not None and np.all(np.isfinite(pcov)):
        perr = np.sqrt(np.diag(pcov))

    result = {
        "fit_model": "exponential",
        "n_points": len(group),
        "time_start_s": float(np.nanmin(t_raw)),
        "time_end_s": float(np.nanmax(t_raw)),
        "y_inf": float(popt[0]),
        "amplitude": float(popt[1]),
        "tau_s": float(popt[2]),
        "tau_err_s": float(perr[2]) if np.isfinite(perr[2]) else np.nan,
        "rmse": rmse,
        "r2": float(r2) if np.isfinite(r2) else np.nan,
        "fit_status": "ok",
    }
    return result, group.assign(t_fit_s=t, y_fit=y_fit), "ok"


def triple_exp_start_guesses(t, y):
    duration = max(float(np.nanmax(t)), 1e-9)
    y_span = max(float(np.nanmax(y) - np.nanmin(y)), 1e-12)
    peak_change = float(np.nanmax(y) - y[0])
    sign = 1.0 if peak_change >= 0 else -1.0
    candidates = [
        ((duration / 200, duration / 20, duration / 2), (0.2, 1.2, -0.4)),
        ((duration / 100, duration / 10, duration), (0.25, 1.25, -0.5)),
        ((duration / 50, duration / 8, duration * 2), (0.5, 1.0, -0.5)),
        ((duration / 20, duration / 4, duration * 4), (0.2, 1.5, -0.8)),
    ]
    guesses = []
    for taus, amplitude_factors in candidates:
        tau1, tau2, tau3 = (max(float(tau), 1e-6) for tau in taus)
        amp1, amp2, amp3 = (factor * y_span * sign for factor in amplitude_factors)
        guesses.append([float(y[0]), amp1, tau1, amp2, tau2, amp3, tau3])
    return guesses


def resample_for_log_time_fit(t, y, n_log_points=300):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    positive_mask = t > 0
    if positive_mask.sum() < 2:
        return t, y

    positive_t = t[positive_mask]
    positive_y = y[positive_mask]
    order = np.argsort(positive_t)
    positive_t = positive_t[order]
    positive_y = positive_y[order]
    unique_t, unique_indices = np.unique(positive_t, return_index=True)
    unique_y = positive_y[unique_indices]

    log_grid = np.logspace(np.log10(unique_t.min()), np.log10(unique_t.max()), n_log_points)
    y_grid = np.interp(log_grid, unique_t, unique_y)

    baseline_y = float(np.nanmedian(y[t <= 0])) if np.any(t <= 0) else float(y[0])
    return np.concatenate([[0.0], log_grid]), np.concatenate([[baseline_y], y_grid])


def fit_triple_exp_saturation(group):
    group = prepare_fit_group(group, after_switch_only=False)
    if len(group) < max(MIN_POINTS_PER_FIT, 10):
        return None, group, "too_few_points"

    t_raw = group["Time_Relative_s"].to_numpy(dtype=float)
    y = group["value"].to_numpy(dtype=float)
    t_min = float(np.nanmin(t_raw))
    t_max = float(np.nanmax(t_raw))
    t = np.clip(t_raw, 0, None)
    t_fit_data, y_fit_data = resample_for_log_time_fit(t, y)

    if np.nanmax(t) <= 0 or np.nanstd(y) == 0:
        return None, group, "not_enough_variation"

    duration = max(float(np.nanmax(t_fit_data)), 1e-9)
    y_span = max(float(np.nanmax(y_fit_data) - np.nanmin(y_fit_data)), 1e-12)
    lower = [
        float(np.nanmin(y_fit_data) - 5 * y_span),
        -10 * y_span,
        1e-9,
        -10 * y_span,
        max(duration / 8, 1e-9),
        -10 * y_span,
        max(duration / 2, 1e-9),
    ]
    upper = [
        float(np.nanmax(y_fit_data) + 5 * y_span),
        10 * y_span,
        max(duration / 8, 1.0),
        10 * y_span,
        max(duration * 2, 1.0),
        10 * y_span,
        max(duration * 20, 1.0),
    ]

    best = None
    failures = []
    for p0 in triple_exp_start_guesses(t_fit_data, y_fit_data):
        p0 = np.minimum(np.maximum(p0, lower), upper)
        try:
            popt_candidate, pcov_candidate = curve_fit(
                triple_exp_saturation_model,
                t_fit_data,
                y_fit_data,
                p0=p0,
                bounds=(lower, upper),
                maxfev=30000,
            )
            residuals_candidate = y_fit_data - triple_exp_saturation_model(t_fit_data, *popt_candidate)
            ss_res_candidate = float(np.sum(residuals_candidate**2))
            if best is None or ss_res_candidate < best[0]:
                best = (ss_res_candidate, popt_candidate, pcov_candidate)
        except Exception as exc:
            failures.append(str(exc))

    if best is None:
        message = failures[-1] if failures else "unknown"
        return None, group, f"fit_failed: {message}"

    _, popt, pcov = best
    y_fit = triple_exp_saturation_model(t, *popt)
    residuals = y - y_fit
    ss_res = float(np.sum(residuals**2))
    ss_tot = float(np.sum((y - np.mean(y))**2))
    r2 = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot
    rmse = float(np.sqrt(np.mean(residuals**2)))

    perr = np.full(7, np.nan)
    if pcov is not None and np.all(np.isfinite(pcov)):
        perr = np.sqrt(np.diag(pcov))

    y0, amplitude1, tau1, amplitude2, tau2, amplitude3, tau3 = popt
    tau1_err = perr[2]
    tau2_err = perr[4]
    tau3_err = perr[6]
    result = {
        "fit_model": "triple_exp_saturation",
        "n_points": len(group),
        "time_start_s": t_min,
        "time_end_s": t_max,
        "y0": float(y0),
        "amplitude1": float(amplitude1),
        "amplitude2": float(amplitude2),
        "amplitude3": float(amplitude3),
        "tau1_s": float(tau1),
        "tau2_s": float(tau2),
        "tau3_s": float(tau3),
        "tau1_err_s": float(tau1_err) if np.isfinite(tau1_err) else np.nan,
        "tau2_err_s": float(tau2_err) if np.isfinite(tau2_err) else np.nan,
        "tau3_err_s": float(tau3_err) if np.isfinite(tau3_err) else np.nan,
        "tau_s": np.nan,
        "tau_err_s": np.nan,
        "rmse": rmse,
        "r2": float(r2) if np.isfinite(r2) else np.nan,
        "fit_status": "ok",
    }
    return result, group.assign(t_fit_s=t, y_fit=y_fit), "ok"


fit_rows = []
fit_curves = []
group_columns = ["Material", "Temperature_C", "Mode", "observable"]
fit_data_long = data_long.copy()
if FIT_TEMPERATURES_C is not None:
    base_fit_mask = fit_data_long["Temperature_C"].isin(FIT_TEMPERATURES_C)
    extra_step_height_abs_mask = (
        fit_data_long["Temperature_C"].isin(EXTRA_STEP_HEIGHT_ABS_TEMPERATURES_C)
        & (fit_data_long["Mode"] == "Abs")
        & (fit_data_long["observable"] == STEP_HEIGHT_OBSERVABLE)
    )
    fit_data_long = fit_data_long[base_fit_mask | extra_step_height_abs_mask]
if FIT_OBSERVABLES is not None:
    fit_data_long = fit_data_long[fit_data_long["observable"].isin(FIT_OBSERVABLES)]
fit_data_long = fit_data_long[
    (fit_data_long["observable"] != OMEGA_P_OBSERVABLE)
    | (fit_data_long["Mode"] == "Abs")
]

for group_key, group in fit_data_long.groupby(group_columns):
    base_row = dict(zip(group_columns, group_key))
    fit_function = fit_exponential
    if should_fit_step_height_des_triple_exp(base_row):
        fit_groups = fit_step_height_des_post_maximum_window(group)
        fit_function = fit_triple_exp_saturation
    elif should_split_step_height_80_abs_to_maximum(base_row):
        fit_groups = split_step_height_80_abs_to_maximum(group)
    elif base_row["observable"] == STEP_HEIGHT_OBSERVABLE:
        fit_groups = split_step_height_group(group, base_row["Mode"])
    elif should_fit_de_50_des_to_3000(base_row):
        fit_groups = fit_until_time(group, DE_50_DES_END_TIME_S)
    elif should_split_de_60_des(base_row):
        fit_groups = split_at_time(group, DE_60_DES_SPLIT_TIME_S)
    elif should_split_de_70_des(base_row):
        fit_groups = split_at_time(group, DE_70_DES_SPLIT_TIME_S)
    elif should_split_de_70_abs(base_row):
        fit_groups = split_at_time(group, DE_70_ABS_SPLIT_TIME_S)
    else:
        fit_groups = [("full", group, {})]

    for fit_segment, segment_group, segment_metadata in fit_groups:
        result, curve, status = fit_function(segment_group)
        row = base_row.copy()
        row.update({
            "fit_segment": fit_segment,
        })
        row.update(segment_metadata)
        if result is None:
            row.update({"fit_status": status, "fit_model": "triple_exp_saturation" if fit_function == fit_triple_exp_saturation else "exponential", "n_points": len(curve), "tau_s": np.nan, "tau_err_s": np.nan, "r2": np.nan, "rmse": np.nan})
        else:
            row.update(result)
            fit_curves.append(curve.assign(**row))
        fit_rows.append(row)

relaxation_results = pd.DataFrame(fit_rows).sort_values(group_columns + ["fit_segment"])
relaxation_results

In [ ]:
def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def fit_group_for_result(result_row):
    mask = (
        (data_long["Material"] == result_row["Material"])
        & (data_long["Temperature_C"] == result_row["Temperature_C"])
        & (data_long["Mode"] == result_row["Mode"])
        & (data_long["observable"] == result_row["observable"])
    )
    return data_long.loc[mask].sort_values("Time_Relative_s")


def plot_group(group, result_row, save=False, x_scale="linear", y_scale="linear", xlim=None, ylim=None, figsize=(6.4, 4.0)):
    material = result_row["Material"]
    temp = result_row["Temperature_C"]
    mode = result_row["Mode"]
    observable = result_row["observable"]
    fit_segment = result_row.get("fit_segment", "full")

    group = group.sort_values("Time_Relative_s").copy()
    t0 = result_row["time_start_s"] if "time_start_s" in result_row and pd.notna(result_row["time_start_s"]) else group["Time_Relative_s"].min()
    if "time_start_s" in result_row and pd.notna(result_row["time_start_s"]):
        group = group[group["Time_Relative_s"] >= result_row["time_start_s"]]
    if "time_end_s" in result_row and pd.notna(result_row["time_end_s"]):
        group = group[group["Time_Relative_s"] <= result_row["time_end_s"]]
    if x_scale == "log":
        group = group[group["Time_Relative_s"] > 0]
    if y_scale == "log":
        group = group[group["value"] > 0]
    if group.empty:
        raise ValueError("Keine Datenpunkte fuer die gewaehlte Darstellung gefunden.")

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(group["Time_Relative_s"], group["value"], s=18, label="Daten")
    split_points = [
        (result_row.get("split_point_1_s", np.nan), result_row.get("split_point_1_type", "Split 1")),
        (result_row.get("split_point_2_s", np.nan), result_row.get("split_point_2_type", "Split 2")),
        (result_row.get("split_point_3_s", np.nan), result_row.get("split_point_3_type", "Split 3")),
    ]
    for split_time, split_type in split_points:
        if pd.notna(split_time):
            if split_type == "minimum":
                label = "Minimum"
            elif split_type == "inflection":
                label = "Wendepunkt"
            elif split_type == "maximum":
                label = "Maximum"
            elif split_type == "kink":
                label = "Knick"
            elif split_type == "manual_time":
                label = f"Split bei {split_time:g} s"
            else:
                label = "Split"
            ax.axvline(split_time, color="tab:gray", linestyle="--", linewidth=1.0, label=label)

    if result_row.get("fit_status") == "ok":
        t_raw = group["Time_Relative_s"].to_numpy(dtype=float)
        t_plot_raw = np.linspace(np.nanmin(t_raw), np.nanmax(t_raw), 300)
        t_plot = t_plot_raw - t0
        if result_row.get("fit_model") == "triple_exp_saturation":
            y_plot = triple_exp_saturation_model(
                np.clip(t_plot_raw, 0, None),
                result_row["y0"],
                result_row["amplitude1"],
                result_row["tau1_s"],
                result_row["amplitude2"],
                result_row["tau2_s"],
                result_row["amplitude3"],
                result_row["tau3_s"],
            )
            fit_label = f"Fit {fit_segment}: tau1 = {result_row['tau1_s']:.3g} s, tau2 = {result_row['tau2_s']:.3g} s, tau3 = {result_row['tau3_s']:.3g} s"
        else:
            y_plot = exp_model(t_plot, result_row["y_inf"], result_row["amplitude"], result_row["tau_s"])
            fit_label = f"Fit {fit_segment}: tau = {result_row['tau_s']:.3g} s"
        ax.plot(t_plot_raw, y_plot, color="tab:red", label=fit_label)

    title_segment = "" if fit_segment == "full" else f" ({fit_segment})"
    title_scale = " | log Zeit" if x_scale == "log" else ""
    ax.set_title(f"{material}, {temp:g} C, {mode}: {observable}{title_segment}{title_scale}")
    ax.set_xlabel("relative Zeit / s")
    ax.set_ylabel(observable)
    if x_scale == "log":
        ax.set_xscale("log")
    elif x_scale == "symlog":
        ax.set_xscale("symlog", linthresh=1.0)
    if y_scale == "log":
        ax.set_yscale("log")
    elif y_scale == "symlog":
        ax.set_yscale("symlog", linthresh=1e-3)
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.legend()
    fig.tight_layout()

    if save:
        PLOT_DIR.mkdir(parents=True, exist_ok=True)
        scale_suffix = "_log_time" if x_scale == "log" else ""
        filename = f"relaxation_{safe_name(material)}_{temp:g}C_{mode}_{safe_name(observable)}_{safe_name(fit_segment)}{scale_suffix}.png"
        fig.savefig(PLOT_DIR / filename, bbox_inches="tight")
    return fig, ax


FIT_PLOT_X_SCALE = "linear"  # Standard fuer normale Fits: "linear", "log" oder "symlog"


def fit_plot_x_scale(result_row):
    if result_row.get("fit_model") == "triple_exp_saturation":
        return "log"
    return FIT_PLOT_X_SCALE


with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    for _, result_row in relaxation_results.iterrows():
        if result_row["fit_status"] != "ok":
            continue
        group = fit_group_for_result(result_row)
        plot_group(group, result_row, save=False, x_scale=fit_plot_x_scale(result_row))
        plt.show()

print("Es wurde nichts gespeichert. Zum Speichern die letzte Zelle verwenden.")

In [ ]:
summary = relaxation_results.pivot_table(
    index=["Material", "Temperature_C", "Mode"],
    columns=["observable", "fit_segment"],
    values="tau_s",
    aggfunc="first",
)
summary

## Tabellarische Uebersicht

Diese Tabelle fasst alle berechneten Relaxationszeiten zusammen. Sie wird nur angezeigt; gespeichert wird weiterhin erst in der letzten Zelle mit `SPEICHERN = True`.

In [ ]:
relaxation_table_columns = [
    "Material",
    "Temperature_C",
    "Mode",
    "observable",
    "fit_segment",
    "fit_model",
    "time_start_s",
    "time_end_s",
    "kink_time_s",
    "n_points",
    "tau_s",
    "tau_err_s",
    "tau1_s",
    "tau1_err_s",
    "tau2_s",
    "tau2_err_s",
    "tau3_s",
    "tau3_err_s",
    "amplitude1",
    "amplitude2",
    "amplitude3",
    "r2",
    "rmse",
    "fit_status",
]

relaxation_times_table = relaxation_results[relaxation_table_columns].copy()
relaxation_times_table = relaxation_times_table.sort_values([
    "Material",
    "Temperature_C",
    "Mode",
    "observable",
    "time_start_s",
    "fit_segment",
])

relaxation_times_table.style.format({
    "Temperature_C": "{:.0f}",
    "time_start_s": "{:.3g}",
    "time_end_s": "{:.3g}",
    "kink_time_s": "{:.3g}",
    "tau_s": "{:.4g}",
    "tau_err_s": "{:.3g}",
    "tau1_s": "{:.4g}",
    "tau1_err_s": "{:.3g}",
    "tau2_s": "{:.4g}",
    "tau2_err_s": "{:.3g}",
    "tau3_s": "{:.4g}",
    "tau3_err_s": "{:.3g}",
    "amplitude1": "{:.4g}",
    "amplitude2": "{:.4g}",
    "amplitude3": "{:.4g}",
    "r2": "{:.4f}",
    "rmse": "{:.4g}",
})

## Abs-Stufenhoehe: Minimum und Stufe danach

Diese Tabelle zeigt fuer `delta epsilon'` aus der Stufenhoehe im Abs-Modus die Zeitkonstante bis zum Minimum und die anschliessende Zeitkonstante nach dem Minimum. Die Spalte `Kommentar` markiert, ob nach dem Minimum eine sichtbare Stufe bzw. ein Wiederanstieg vorliegt.

In [ ]:
stepheight_abs = relaxation_results[
    (relaxation_results["observable"] == "delta_eps_real_step_height")
    & (relaxation_results["Mode"] == "Abs")
    & (relaxation_results["fit_status"] == "ok")
].copy()

stepheight_abs_tau = stepheight_abs.pivot_table(
    index=["Material", "Temperature_C"],
    columns="fit_segment",
    values="tau_s",
    aggfunc="first",
).reset_index()

stepheight_abs_ranges = stepheight_abs.pivot_table(
    index=["Material", "Temperature_C"],
    columns="fit_segment",
    values=["time_start_s", "time_end_s", "r2"],
    aggfunc="first",
).reset_index()
stepheight_abs_ranges.columns = ["_".join(str(part) for part in column if part) for column in stepheight_abs_ranges.columns]

stepheight_abs_diagnostics = stepheight_abs_tau.merge(stepheight_abs_ranges, on=["Material", "Temperature_C"], how="left")

def comment_post_minimum_step(row):
    tau_after = row.get("from_minimum", np.nan)
    tau_to_min = row.get("to_minimum", np.nan)
    if pd.isna(tau_after):
        return "keine Nach-Minimum-Zeitkonstante"
    if pd.notna(tau_to_min) and tau_to_min < 100 and tau_after > 500:
        return "schneller Abfall bis Minimum, danach deutlich langsamere Stufe/Wiederanstieg"
    if tau_after > 500:
        return "langsame Stufe/Wiederanstieg nach dem Minimum"
    return "Nach-Minimum-Stufe eher schnell/schwach"

stepheight_abs_diagnostics["Kommentar"] = stepheight_abs_diagnostics.apply(comment_post_minimum_step, axis=1)
stepheight_abs_diagnostics.style.format({
    "Temperature_C": "{:.0f}",
    "to_minimum": "{:.4g}",
    "from_minimum": "{:.4g}",
    "time_start_s_to_minimum": "{:.3g}",
    "time_end_s_to_minimum": "{:.3g}",
    "time_start_s_from_minimum": "{:.3g}",
    "time_end_s_from_minimum": "{:.3g}",
    "r2_to_minimum": "{:.4f}",
    "r2_from_minimum": "{:.4f}",
})

## Arrhenius-Darstellung

Hier werden die Relaxationszeiten von `delta epsilon'` aus der Stufenhoehe fuer Absorption nach dem Minimum und die Zeitkonstanten aus `omega_p` fuer Absorption gegen die inverse Temperatur dargestellt. Die Stepheight-Serie enthaelt 50, 60, 70, 80 und 90 C; `omega_p` enthaelt 50, 60 und 70 C. Die Fit-Parameter `de` sind hier bewusst nicht enthalten. Im Plot ist `tau` logarithmisch gegen `1000/T` aufgetragen.

In [ ]:
R_GAS_CONSTANT = 8.314462618  # J / (mol K)
ARRHENIUS_STEPHEIGHT_TEMPERATURES_C = [50, 60, 70, 80, 90]
ARRHENIUS_OMEGA_P_TEMPERATURES_C = [50, 60, 70]

arrhenius_stepheight = relaxation_results[
    (relaxation_results["observable"] == "delta_eps_real_step_height")
    & (relaxation_results["Mode"] == "Abs")
    & (relaxation_results["fit_segment"] == "from_minimum")
    & (relaxation_results["fit_status"] == "ok")
    & (relaxation_results["tau_s"] > 0)
    & (relaxation_results["Temperature_C"].isin(ARRHENIUS_STEPHEIGHT_TEMPERATURES_C))
].copy()
arrhenius_stepheight["arrhenius_series"] = "delta epsilon' Stepheight, Abs, nach Minimum"

arrhenius_omega_p = relaxation_results[
    (relaxation_results["observable"] == "omega_p")
    & (relaxation_results["Mode"] == "Abs")
    & (relaxation_results["fit_status"] == "ok")
    & (relaxation_results["tau_s"] > 0)
    & (relaxation_results["Temperature_C"].isin(ARRHENIUS_OMEGA_P_TEMPERATURES_C))
].copy()
arrhenius_omega_p["arrhenius_series"] = "omega_p, Abs"

arrhenius_data = pd.concat([arrhenius_stepheight, arrhenius_omega_p], ignore_index=True)
arrhenius_data["Temperature_K"] = arrhenius_data["Temperature_C"] + 273.15
arrhenius_data["inverse_temperature_1_per_K"] = 1 / arrhenius_data["Temperature_K"]
arrhenius_data["inverse_temperature_1000_per_K"] = 1000 / arrhenius_data["Temperature_K"]
arrhenius_data["ln_tau"] = np.log(arrhenius_data["tau_s"])
arrhenius_data = arrhenius_data.sort_values(["arrhenius_series", "inverse_temperature_1_per_K"])

fig, ax = plt.subplots(figsize=(7.2, 4.6))
arrhenius_fit_rows = []

if arrhenius_data.empty:
    ax.text(0.5, 0.5, "Keine passenden Relaxationszeiten gefunden.", ha="center", va="center", transform=ax.transAxes)
else:
    for series_name, series_data in arrhenius_data.groupby("arrhenius_series"):
        series_data = series_data.sort_values("inverse_temperature_1_per_K")
        ax.scatter(
            series_data["inverse_temperature_1000_per_K"],
            series_data["tau_s"],
            s=45,
            label=series_name,
        )

        if len(series_data) >= 2:
            slope, intercept = np.polyfit(series_data["inverse_temperature_1_per_K"], series_data["ln_tau"], 1)
            ea_j_per_mol = slope * R_GAS_CONSTANT
            ea_kj_per_mol = ea_j_per_mol / 1000
            tau0_s = np.exp(intercept)
            x_fit = np.linspace(
                series_data["inverse_temperature_1_per_K"].min(),
                series_data["inverse_temperature_1_per_K"].max(),
                100,
            )
            tau_fit = np.exp(intercept + slope * x_fit)
            ax.plot(1000 * x_fit, tau_fit, label=f"Arrhenius-Fit {series_name}: Ea = {ea_kj_per_mol:.2f} kJ/mol")
            arrhenius_fit_rows.append(
                {
                    "arrhenius_series": series_name,
                    "observable": series_data["observable"].iloc[0],
                    "mode": series_data["Mode"].iloc[0],
                    "fit_segment": ", ".join(series_data["fit_segment"].dropna().astype(str).unique()),
                    "n_points": len(series_data),
                    "activation_energy_kJ_per_mol": ea_kj_per_mol,
                    "tau0_s": tau0_s,
                    "slope_K": slope,
                    "intercept": intercept,
                }
            )

ax.set_yscale("log")
ax.set_xlabel("1000 / T / 1/K")
ax.set_ylabel("Relaxationszeit tau / s")
ax.set_title("Arrhenius-Plot: Stepheight und omega_p, Abs")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
plt.show()

arrhenius_fit_table = pd.DataFrame(arrhenius_fit_rows)
arrhenius_display = arrhenius_data[
    [
        "arrhenius_series",
        "Material",
        "Temperature_C",
        "Temperature_K",
        "inverse_temperature_1000_per_K",
        "tau_s",
        "fit_segment",
        "fit_model",
        "r2",
    ]
].reset_index(drop=True)

display(arrhenius_display)
display(arrhenius_fit_table)


## Fit interaktiv anschauen

Die vorherige Plot-Zelle zeigt weiterhin automatisch alle erfolgreichen Fits an. In diesem zusaetzlichen Abschnitt kannst du danach einen einzelnen Fit auswaehlen und die Darstellung genauer anpassen. Das dient nur zur Kontrolle; gespeichert wird weiterhin erst in der letzten Zelle mit `SPEICHERN = True`.

In [ ]:
def parse_optional_float(value):
    value = str(value).strip()
    return None if value == "" else float(value)


def parse_optional_limit(min_value, max_value):
    lower = parse_optional_float(min_value)
    upper = parse_optional_float(max_value)
    if lower is None and upper is None:
        return None
    return (lower, upper)


def format_fit_option(index, row):
    model = row.get("fit_model", "exponential")
    r2 = row.get("r2", np.nan)
    r2_text = "nan" if pd.isna(r2) else f"{r2:.4f}"
    return (
        f"{index}: {row['Material']}, {row['Temperature_C']:g} C, {row['Mode']}, "
        f"{row['observable']}, {row['fit_segment']}, {model}, R2={r2_text}"
    )


def show_selected_fit(result_index, x_scale="linear", y_scale="linear", x_min="", x_max="", y_min="", y_max="", width=7.0, height=4.4):
    result_row = relaxation_results.loc[result_index]
    group = fit_group_for_result(result_row)
    xlim = parse_optional_limit(x_min, x_max)
    ylim = parse_optional_limit(y_min, y_max)
    fig, ax = plot_group(
        group,
        result_row,
        save=False,
        x_scale=x_scale,
        y_scale=y_scale,
        xlim=xlim,
        ylim=ylim,
        figsize=(float(width), float(height)),
    )
    plt.show()

    detail_columns = [
        "Material", "Temperature_C", "Mode", "observable", "fit_segment", "fit_model",
        "time_start_s", "time_end_s", "n_points", "tau_s", "tau1_s", "tau2_s", "tau3_s", "amplitude1", "amplitude2", "amplitude3", "r2", "rmse", "fit_status",
    ]
    detail_columns = [column for column in detail_columns if column in relaxation_results.columns]
    display(result_row[detail_columns].to_frame("Wert"))


ok_fit_options = relaxation_results[relaxation_results["fit_status"] == "ok"].copy()
if ok_fit_options.empty:
    print("Keine erfolgreichen Fits zum Anzeigen vorhanden.")
else:
    try:
        import ipywidgets as widgets
        from IPython.display import clear_output, display

        option_pairs = [(format_fit_option(index, row), index) for index, row in ok_fit_options.iterrows()]
        fit_dropdown = widgets.Dropdown(options=option_pairs, description="Fit", layout=widgets.Layout(width="95%"))
        x_scale_dropdown = widgets.Dropdown(options=["linear", "log", "symlog"], value="linear", description="x-Skala")
        y_scale_dropdown = widgets.Dropdown(options=["linear", "log", "symlog"], value="linear", description="y-Skala")
        x_min_text = widgets.Text(value="", description="x min", placeholder="auto")
        x_max_text = widgets.Text(value="", description="x max", placeholder="auto")
        y_min_text = widgets.Text(value="", description="y min", placeholder="auto")
        y_max_text = widgets.Text(value="", description="y max", placeholder="auto")
        width_box = widgets.FloatText(value=7.0, description="Breite")
        height_box = widgets.FloatText(value=4.4, description="Hoehe")
        update_button = widgets.Button(description="Plot anzeigen", button_style="primary")
        output = widgets.Output()

        def update_plot(_=None):
            with output:
                clear_output(wait=True)
                try:
                    show_selected_fit(
                        fit_dropdown.value,
                        x_scale=x_scale_dropdown.value,
                        y_scale=y_scale_dropdown.value,
                        x_min=x_min_text.value,
                        x_max=x_max_text.value,
                        y_min=y_min_text.value,
                        y_max=y_max_text.value,
                        width=width_box.value,
                        height=height_box.value,
                    )
                except Exception as exc:
                    print(f"Plot konnte nicht erstellt werden: {exc}")

        update_button.on_click(update_plot)
        controls = widgets.VBox([
            fit_dropdown,
            widgets.HBox([x_scale_dropdown, y_scale_dropdown]),
            widgets.HBox([x_min_text, x_max_text, y_min_text, y_max_text]),
            widgets.HBox([width_box, height_box, update_button]),
            output,
        ])
        display(controls)
        update_plot()
    except ImportError:
        print("ipywidgets ist nicht verfuegbar. Nutze stattdessen diese manuelle Auswahl:")
        display(pd.DataFrame(option_pairs, columns=["Fit", "Index"]))
        RESULT_INDEX = ok_fit_options.index[0]
        X_SCALE = "linear"  # "linear", "log" oder "symlog"
        Y_SCALE = "linear"  # "linear", "log" oder "symlog"
        X_LIM = None  # z.B. (-100, 500) oder None
        Y_LIM = None  # z.B. (2.0, 3.0) oder None
        FIGSIZE = (7.0, 4.4)
        result_row = relaxation_results.loc[RESULT_INDEX]
        plot_group(fit_group_for_result(result_row), result_row, x_scale=X_SCALE, y_scale=Y_SCALE, xlim=X_LIM, ylim=Y_LIM, figsize=FIGSIZE)
        plt.show()


## Optional speichern

Erst wenn die Ergebnisse plausibel aussehen, `SPEICHERN = True` setzen und diese Zelle ausfuehren.

In [ ]:
SPEICHERN = False

if SPEICHERN:
    output_csv = RESULTS_DIR / "relaxation_times_exponential_fits.csv"
    relaxation_results.to_csv(output_csv, index=False)

    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    saved_plots = 0
    for _, result_row in relaxation_results.iterrows():
        if result_row["fit_status"] != "ok":
            continue
        group = fit_group_for_result(result_row)
        fig, _ = plot_group(group, result_row, save=True, x_scale=fit_plot_x_scale(result_row))
        plt.close(fig)
        saved_plots += 1

    print(f"Gespeichert: {output_csv}")
    print(f"Gespeicherte Plots: {saved_plots} in {PLOT_DIR}")
else:
    print("Nichts gespeichert. Setze SPEICHERN = True, wenn du die Ergebnisse exportieren willst.")